## baseline_model_colab
- Role: train one simple model and report the result.
- I used the leakage-safe train/test split from the notebook.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

raw_path = Path('disease_symptoms_raw.csv')
if not raw_path.exists():
    raw_path = Path('data/raw/disease_symptoms_raw.csv')

print('raw_path:', raw_path)
raw_df = pd.read_csv(raw_path)
raw_df.head()


### Clean columns
- I clean the column names before splitting the file.


In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    drop_cols = [c for c in df.columns if c == '' or c.lower().startswith('unnamed')]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

raw_df = clean_columns(raw_df)
target = 'prognosis'
symptom_cols = [c for c in raw_df.columns if c != target]
print('Raw shape:', raw_df.shape)


### Split raw data
- I split by unique symptom pattern so the same pattern does not appear in both train and test.


In [ ]:
from sklearn.model_selection import train_test_split

raw_df['_symptom_signature'] = raw_df[symptom_cols].astype(str).agg('|'.join, axis=1)
unique_patterns = raw_df[[target, '_symptom_signature']].drop_duplicates()

test_signatures = []
for label, group in unique_patterns.groupby(target):
    _, label_test = train_test_split(
        group['_symptom_signature'],
        test_size=0.2,
        random_state=42,
    )
    test_signatures.extend(label_test.tolist())

test_signature_set = set(test_signatures)
train_df = raw_df[~raw_df['_symptom_signature'].isin(test_signature_set)].copy()
test_df = raw_df[raw_df['_symptom_signature'].isin(test_signature_set)].copy()

output_dir = Path('data')
output_dir.mkdir(exist_ok=True)
train_output = output_dir / 'Training.csv'
test_output = output_dir / 'Testing.csv'

train_df = train_df.drop(columns=['_symptom_signature'])
test_df = test_df.drop(columns=['_symptom_signature'])
train_df.to_csv(train_output, index=False)
test_df.to_csv(test_output, index=False)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Saved:', train_output)
print('Saved:', test_output)


### Leakage check
- I check that train and test do not share the same symptom pattern.


In [ ]:
train_signatures = set(train_df[symptom_cols].astype(str).agg('|'.join, axis=1))
test_signatures = set(test_df[symptom_cols].astype(str).agg('|'.join, axis=1))
print('Shared symptom patterns:', len(train_signatures & test_signatures))


### Baseline model
- I used Bernoulli Naive Bayes because the symptom columns are 0 and 1.


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

X_train = train_df[symptom_cols]
X_test = test_df[symptom_cols]
y_train_text = train_df[target].astype(str)
y_test_text = test_df[target].astype(str)

le = LabelEncoder()
y_train = le.fit_transform(y_train_text)
y_test = le.transform(y_test_text)

model = BernoulliNB()
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, pred), 4))

sample = X_test.iloc[[0]]
proba = model.predict_proba(sample)[0]
top3 = np.argsort(proba)[::-1][:3]
print('Top-3 predictions for 1 sample:')
for i in top3:
    print('-', le.inverse_transform([i])[0], f'{proba[i]*100:.1f}%')


### Short note
- This split removes the clear data leakage problem from duplicate rows.
- If the score is still very high, that comes from the dataset being very clean and easy to separate.
